In [1]:
pip install datasets

  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
   ---------------------------------------- 0.0/608.3 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/608.3 kB ? eta -:--:--
   ---------------------------------- ----- 524.3/608.3 kB 1.0 MB/s eta 0:00:01
   ---------------------------------------- 608.3/608.3 kB 886.3 kB/s  0:00:00
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
   ---------------------------------------- 0.0/29.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/29.8 MB ? eta -:--:--
    --------------------------------------- 0.5/29.8 MB 1.3 MB/s eta 0:00:24
   - -------------------------------------- 0.8/29.8 MB 1.4 MB/s eta 0:00:21
   - -------------------------------------- 1.0/29.8 MB 1.1 MB/s eta 0:00:26
   - -------------------------------------- 1.0/29.8 MB 1.1 MB/s eta 0:00:26
   - -------------------------------------- 1.3/29.8 MB 1.0 MB/s eta 0:00:28
   -- ------------------------------------- 1.6/29.8 MB 1.1 MB

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from datasets import load_dataset

dataset = load_dataset("tweet_eval", "sentiment", split="test")
print(dataset)

c:\Users\excel\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using the latest cached version of the dataset since tweet_eval couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'sentiment' at C:\Users\excel\.cache\huggingface\datasets\tweet_eval\sentiment\0.0.0\b3a375baf0f409c77e6bc7aa35102b7b3534f8be (last modified on Mon Nov 23 16:47:44 2026).


Dataset({
    features: ['text', 'label'],
    num_rows: 12284
})


In [3]:
samples = dataset.select(range(100))
print(samples[0])

{'text': "@user @user what do these '1/2 naked pics' have to do with anything? They're not even like that.", 'label': 1}


In [4]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis",model="cardiffnlp/twitter-roberta-base-sentiment-latest")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8721.86it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
texts = list(samples["text"])
results = classifier(texts, truncation=True)
print(results[0])

{'label': 'negative', 'score': 0.8047260642051697}


In [6]:
names = ["negative", "neutral", "positive"]

In [7]:
correct = 0
for label, result in zip(samples["label"], results):
    if result["label"] == names[label]:
        correct += 1

print("Accuracy:", correct / len(samples))

Accuracy: 0.74


In [8]:
for i in range(5):
    print(samples["text"][i])
    print("Prediction:", results[i]["label"], round(results[i]["score"], 4))
    print("Actual:", names[samples["label"][i]])
    print()

@user @user what do these '1/2 naked pics' have to do with anything? They're not even like that.
Prediction: negative 0.8047
Actual: neutral

OH: “I had a blue penis while I was this” [playing with Google Earth VR]
Prediction: neutral 0.8669
Actual: neutral

@user @user That's coming, but I think the victims are going to be Medicaid recipients.
Prediction: neutral 0.7637
Actual: neutral

I think I may be finally in with the in crowd #mannequinchallenge  #grads2014 @user
Prediction: positive 0.774
Actual: positive

@user Wow,first Hugo Chavez and now Fidel Castro. Danny Glover, Michael Moore, Oliver Stone, and Sean Penn are running out of heroes.
Prediction: neutral 0.4164
Actual: negative



In [9]:
from sklearn.metrics import classification_report

y_true = [names[l] for l in samples["label"]]
y_pred = [r["label"] for r in results]
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

    negative       0.61      0.79      0.69        29
     neutral       0.88      0.65      0.75        55
    positive       0.71      0.94      0.81        16

    accuracy                           0.74       100
   macro avg       0.73      0.80      0.75       100
weighted avg       0.77      0.74      0.74       100



<hr>

<h3 style="text-align:center; color:green;">
    End of Assignment
</h3>

<hr>